# Huấn luyện độ mặn với nested OOF theo thời gian

## Câu hỏi nghiên cứu

Hiệu chỉnh phần dư ngoài fold bằng Gaussian Process thay đổi sai số trên những ngày quan trắc tiếp theo như thế nào? Dùng CSV Landsat 8 đã xử lý trong mục đọc dữ liệu. Đọc bảng cuối theo từng cặp cây và cây + GP trên cùng tập test. Các cột Base_CV mô tả mô hình nền.

Notebook huấn luyện lại cây, tạo **nested forward OOF**, fit GP cộng cơ sở và xuất bộ model dùng cho ba notebook tác động của lambda, kernel và tổng hợp.

- Train: từ 01/01/2019 đến trước 01/03/2020. Test gốc: từ 01/03/2020 trở đi.
- Mỗi outer fold chỉ dùng quá khứ để chọn tham số PSO bằng inner forward CV, fit cây rồi dự báo khối ngày phía sau. Các ngày warm-up giữ `NaN`.
- Mặc định `BASE_MODE='tree_only'`: GP học `y - clip(oof_tree, 0, None)`, khớp nền cây thuần của ba notebook thí nghiệm. Có thể chọn `tree_ridge` để chạy lại nền blend; khi đó GP lưu sẽ học residual blend.
- GP cơ sở là **xu hướng + mùa vụ + không gian + nhiễu**, chưa có tương tác. Các notebook thí nghiệm sẽ thêm `gamma * K_space_int * (K_season_int + K_trend_int)`.
- Dữ liệu CSV là dữ liệu đầu vào có sẵn. Nested OOF kiểm soát việc fit/tuning trong notebook; không xác nhận được các bước chuẩn hóa/chọn biến từng thực hiện trước khi tạo CSV.

## Chạy trên Kaggle

1. Add Input CSV `Check_BMA_Chuanhoa_LS8_20192020_159.csv`, bật Internet và chọn CPU. Không cần PKL/JSON cũ.
2. Run All từ đầu. Ba cell mục 0 tạo Python **3.12.13** riêng; các cell `%%gp312` dùng chung phiên Python đó, không đổi Python của giao diện Kaggle.
3. PSO chạy riêng trong từng outer fold và một lần cho mô hình cuối; đây là phần nặng cần chạy trên Kaggle.
4. Cell **Xuất bộ model** lưu PKL, JSON và OOF vào `/kaggle/working/training_runs/<lần_chạy>/` và tạo ZIP để tải về. Model được lưu trước các biểu đồ và ablation tùy chọn.
5. Dùng PKL và JSON **cùng một lần chạy**, cùng CSV gốc cho các notebook thí nghiệm. Tên file JSON chuẩn là `parameters.json`.

Kết quả test chỉ phục vụ đánh giá; không dùng để chọn tham số PSO, kernel GP hoặc trọng số blend. `Base_CV_*` là điểm OOF của mô hình nền, chưa phải CV của toàn bộ cây + GP.


## 0.1. Chọn môi trường huấn luyện
Tạo Python 3.12.13 riêng, không cần PKL/JSON cũ. Các phiên bản tính toán được cố định để có thể nạp lại model ở notebook thí nghiệm.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys

# Môi trường cố định, tương thích bộ nạp Python 3.12 trong các notebook thí nghiệm.
GP312_ROOT = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()) / 'salinity_python312'
gp312_version = '3.12.13'
gp312_specs = [
    'numpy==2.0.2', 'pandas==2.3.3', 'scipy==1.16.3',
    'scikit-learn==1.6.1', 'xgboost==3.2.0', 'lightgbm==4.6.0',
    'pyswarm==1.1.2', 'cloudpickle==3.1.2',
    'ipykernel==6.29.5', 'matplotlib>=3.9,<3.11', 'seaborn==0.13.2',
    'joblib', 'threadpoolctl',
]
print('Python giao diện:', platform.python_version())
print('Python huấn luyện:', gp312_version)


## 0.2. Cài môi trường riêng
Chỉ lần đầu trong session cần tải Python và thư viện. Bật Internet trên Kaggle; chưa có huấn luyện ở bước này.

In [ ]:
def gp312_run(arguments, env=None):
    print("+", " ".join(str(item) for item in arguments), flush=True)
    subprocess.run([str(item) for item in arguments], check=True, env=env)

GP312_ROOT.mkdir(parents=True, exist_ok=True)
gp312_bootstrap = GP312_ROOT / "uv_bootstrap"
gp312_uv_env = os.environ.copy()
gp312_uv_env["PYTHONPATH"] = str(gp312_bootstrap)
gp312_uv_env["UV_CACHE_DIR"] = str(GP312_ROOT / "uv_cache")
gp312_uv_env["UV_PYTHON_INSTALL_DIR"] = str(GP312_ROOT / "python_versions")
if not (gp312_bootstrap / "uv" / "__main__.py").is_file():
    gp312_run([sys.executable, "-m", "pip", "install", "--target", gp312_bootstrap,
               "--only-binary=:all:", "uv"])
gp312_uv = [sys.executable, "-m", "uv"]
gp312_venv = GP312_ROOT / "venv"
gp312_python = gp312_venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not gp312_python.is_file():
    gp312_run(gp312_uv + ["venv", "--python", gp312_version, "--managed-python", gp312_venv],
               env=gp312_uv_env)
gp312_actual_version = subprocess.check_output(
    [str(gp312_python), "-I", "-c", "import platform; print(platform.python_version())"],
    text=True).strip()
if gp312_actual_version != gp312_version:
    raise RuntimeError("Môi trường đã có khác phiên bản cần dùng; đổi GP312_ROOT rồi chạy lại.")
gp312_signature = hashlib.sha256(json.dumps([gp312_version, gp312_specs]).encode()).hexdigest()
gp312_marker = GP312_ROOT / "environment_ready.json"
gp312_ready = json.loads(gp312_marker.read_text()) if gp312_marker.is_file() else {}
if gp312_ready.get("signature") != gp312_signature:
    gp312_run(gp312_uv + ["pip", "install", "--python", gp312_python] + gp312_specs, env=gp312_uv_env)
    gp312_marker.write_text(json.dumps({"signature": gp312_signature}), encoding="utf-8")
print("Môi trường thí nghiệm đã sẵn sàng:", gp312_python)

## 0.3. Kết nối phiên Python huấn luyện
Các cell `%%gp312` giữ biến trong cùng một tiến trình Python riêng, hiển thị bảng và biểu đồ tại đây. Chạy lại cell này sẽ xóa biến của phiên phụ; sau đó phải chạy lại các cell bên dưới.

In [ ]:
GP312_WORKER_SOURCE = '"""Embedded in notebook setup cells; persistent isolated Python worker."""\nimport contextlib\nimport json\nimport sys\n\nprotocol_output = sys.stdout\nrequest_id = None\n\n\ndef send(kind, **content):\n    message = {"id": request_id, "kind": kind, **content}\n    protocol_output.write(json.dumps(message, ensure_ascii=True, default=str) + "\\n")\n    protocol_output.flush()\n\n\nclass ForwardStream:\n    def __init__(self, name):\n        self.name = name\n\n    def write(self, text):\n        if text:\n            send("stream", name=self.name, text=text)\n        return len(text)\n\n    def flush(self):\n        pass\n\n    @property\n    def encoding(self):\n        return "utf-8"\n\n\nwith contextlib.redirect_stdout(ForwardStream("stdout")), contextlib.redirect_stderr(ForwardStream("stderr")):\n    import matplotlib\n    matplotlib.use("module://matplotlib_inline.backend_inline")\n    from IPython.core.interactiveshell import InteractiveShell\n    from IPython.utils.capture import capture_output\n    from matplotlib_inline.backend_inline import configure_inline_support\n    from traitlets.config import Config\n\n    config = Config()\n    config.HistoryManager.enabled = False\n    shell = InteractiveShell.instance(config=config)\n    configure_inline_support(shell, "module://matplotlib_inline.backend_inline")\n\nsend("ready", python=sys.version)\nfor line in sys.stdin:\n    try:\n        request = json.loads(line)\n        request_id = request["id"]\n        with capture_output(stdout=False, stderr=False) as captured:\n            with contextlib.redirect_stdout(ForwardStream("stdout")), contextlib.redirect_stderr(ForwardStream("stderr")):\n                result = shell.run_cell(request["code"], store_history=True)\n        for output in captured.outputs:\n            send("display_data", data=output.data, metadata=output.metadata)\n        if result.result is not None:\n            data, metadata = shell.display_formatter.format(result.result)\n            send("display_data", data=data, metadata=metadata)\n        error = result.error_before_exec or result.error_in_exec\n        send("done", ok=error is None,\n             error="" if error is None else f"{type(error).__name__}: {error}")\n    except BaseException as error:\n        send("done", ok=False, error=f"{type(error).__name__}: {error}")\n'

import atexit
import signal
from IPython.display import publish_display_data

def gp312_stop():
    worker = globals().get("gp312_worker")
    if worker is not None:
        if worker.poll() is None:
            worker.terminate()
            try:
                worker.wait(timeout=5)
            except subprocess.TimeoutExpired:
                worker.kill()
                worker.wait(timeout=5)
        for stream in (worker.stdin, worker.stdout):
            if stream is not None:
                stream.close()
    log = globals().get("gp312_worker_log_handle")
    if log is not None and not log.closed:
        log.close()

def gp312_show(message):
    kind = message["kind"]
    if kind == "stream":
        stream = sys.stderr if message["name"] == "stderr" else sys.stdout
        stream.write(message["text"])
        stream.flush()
    elif kind == "display_data":
        publish_display_data(data=message["data"], metadata=message.get("metadata", {}))

def gp312_read():
    while True:
        line = gp312_worker.stdout.readline()
        if not line:
            raise RuntimeError(f"Python phụ đã dừng. Xem log {gp312_worker_log_path}. "
                               "Chạy lại cell 0.3 rồi các cell thí nghiệm theo thứ tự.")
        try:
            return json.loads(line)
        except json.JSONDecodeError:
            # Một thư viện native có thể ghi trực tiếp ra stdout, ngoài giao thức.
            print(line, end="", flush=True)

def gp312_execute(code):
    global gp312_request_id
    if gp312_worker.poll() is not None:
        raise RuntimeError("Python phụ đã dừng; chạy lại cell 0.3 trước.")
    gp312_request_id += 1
    current_id = gp312_request_id
    gp312_worker.stdin.write(json.dumps({"id": current_id, "code": code}) + "\n")
    gp312_worker.stdin.flush()
    try:
        while True:
            message = gp312_read()
            if message.get("id") != current_id:
                continue
            if message["kind"] == "done":
                if not message["ok"]:
                    raise RuntimeError("Cell Python phụ lỗi: " + message["error"])
                return
            gp312_show(message)
    except KeyboardInterrupt:
        if os.name == "posix" and gp312_worker.poll() is None:
            gp312_worker.send_signal(signal.SIGINT)
        raise

def gp312_magic(line, cell):
    if line.strip():
        raise ValueError("Dùng %%gp312 trên một dòng riêng, không có tham số.")
    gp312_execute(cell)

# Truyền code và output qua pipe; không khởi chạy KernelManager/asyncio của phiên chính.
gp312_stop()
gp312_runtime = GP312_ROOT / "worker"
gp312_runtime.mkdir(parents=True, exist_ok=True)
gp312_worker_path = gp312_runtime / "worker.py"
gp312_worker_path.write_text(GP312_WORKER_SOURCE, encoding="utf-8")
gp312_worker_log_path = gp312_runtime / "worker_stderr.log"
gp312_worker_log_handle = gp312_worker_log_path.open("w", encoding="utf-8")
gp312_child_env = os.environ.copy()
gp312_child_env.pop("PYTHONPATH", None)
gp312_child_env["OMP_NUM_THREADS"] = "2"
gp312_child_env["OPENBLAS_NUM_THREADS"] = "2"
gp312_child_env["MKL_NUM_THREADS"] = "2"
gp312_child_env["MPLCONFIGDIR"] = str(GP312_ROOT / "matplotlib")
gp312_worker = subprocess.Popen(
    [str(gp312_python), "-I", "-u", str(gp312_worker_path)],
    cwd=str(Path.cwd()), env=gp312_child_env,
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=gp312_worker_log_handle,
    text=True, encoding="utf-8", bufsize=1,
)
gp312_request_id = 0
try:
    while True:
        message = gp312_read()
        if message["kind"] == "ready":
            print("Python phụ đã sẵn sàng:", message["python"])
            break
        gp312_show(message)
    get_ipython().register_magic_function(gp312_magic, magic_kind="cell", magic_name="gp312")
    atexit.register(gp312_stop)
    gp312_execute("import sys\n"
                    + f"assert sys.version.split()[0] == {gp312_version!r}\n"
                    + "print('Python chạy các cell thí nghiệm:', sys.version)")
except Exception:
    gp312_stop()
    raise
print("Đã kết nối. Tiếp tục chạy các cell có dòng %%gp312 bên dưới.")

**Cell 2: Import thư viện & Định nghĩa các hàm đo lường hiệu năng chuẩn của bài báo**
- R2 (Coefficient of Determination - Hệ số xác định)
- RMSE (Root Mean Square Error - Sai số bình phương trung bình căn, đơn vị: g/l hoặc ‰)
- MAE (Mean Absolute Error - Sai số tuyệt đối trung bình, đơn vị: g/l hoặc ‰)
- Ex_VAR (Explained Variance - Phương sai giải thích, đơn vị: %)

In [ ]:
%%gp312
NOTEBOOK_TEMPLATE_CODE_SHA256 = '186daaae1dc982bdabc6776a2768344d595a6302893efd2302ef314515dabe8c'
from pathlib import Path
import os
import re
import copy
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.base import clone
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel
from pyswarm import pso
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)

# 1. Hệ số xác định R2 (Công thức 7 bài báo)
def calc_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

# 2. Sai số bình phương trung bình căn (RMSE - g/l) (Công thức 8 bài báo)
def calc_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# 3. Sai số tuyệt đối trung bình (MAE - g/l) (Công thức 9 bài báo)
def calc_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# 4. Phương sai giải thích (Ex_VAR - %) (Công thức 10 bài báo)
def calc_explained_variance(y_true, y_pred):
    var_y = np.var(y_true)
    if var_y == 0:
        return 0.0
    return (1 - np.var(y_true - y_pred) / var_y) * 100

# 5. Độ lệch trung bình (Bias = mean(y_true - y_pred)); DƯƠNG nghĩa là mô hình đoán THẤP hơn thực tế
def calc_bias(y_true, y_pred):
    return np.mean(y_true - y_pred)

# 6. Hàm tổng hợp các chỉ số. LUÔN gọi theo thứ tự (y_true, y_pred)
def evaluate_model(y_true, y_pred):
    return {
        'RMSE (‰)': round(calc_rmse(y_true, y_pred), 3),
        'MAE (‰)': round(calc_mae(y_true, y_pred), 3),
        'R2': round(calc_r2(y_true, y_pred), 3),
        'Ex_VAR (%)': round(calc_explained_variance(y_true, y_pred), 3),
        'Bias (‰)': round(calc_bias(y_true, y_pred), 3)
    }

print('Import thư viện và định nghĩa các hàm đo lường hoàn tất.')

**Cell 3: Đọc dữ liệu thực tế, giải mã 37 cột ngày và tạo các đặc trưng chu kỳ thời gian tuần hoàn**
- Quét và lọc 37 cột ngày nhị phân cũ (`19Th0119`, `04Th0219`...).
- Phát hiện và loại bỏ 22 dòng khuyết thông tin ngày quan trắc (toàn bộ 37 cột bằng 0).
- Giải mã tên cột ra ngày chuẩn `date`, trích xuất `doy` (Day of Year: 1 - 365), `year`, `month`.
- Tạo 2 biến tuần hoàn `sin_doy` và `cos_doy` cho mô hình cây.
- Xóa sạch 37 cột nhị phân cũ để tối ưu hóa bộ nhớ và triệt tiêu nguy cơ học vẹt.

**3.1 Chia dữ liệu theo thời gian**

In [ ]:
%%gp312

# Nếu có nhiều CSV trùng tên trong Input, điền đường dẫn đầy đủ ở đây.
DATA_PATH = None  # Ví dụ: '/kaggle/input/ten-dataset/Check_BMA_Chuanhoa_LS8_20192020_159.csv'
data_name = 'Check_BMA_Chuanhoa_LS8_20192020_159.csv'
if DATA_PATH is not None:
    source = Path(DATA_PATH)
else:
    local = [Path(data_name), Path('result_csv') / data_name]
    matches = [p.resolve() for p in local if p.is_file()]
    if not matches and Path('/kaggle/input').is_dir():
        matches = list(Path('/kaggle/input').rglob(data_name))
    matches = list(dict.fromkeys(matches))
    if len(matches) != 1:
        raise FileNotFoundError(f'Tìm thấy {len(matches)} CSV. Điền DATA_PATH chính xác: {matches}')
    source = matches[0]
if not source.is_file():
    raise FileNotFoundError(source)
csv_file = str(source.resolve())
print('Đang đọc:', csv_file)
df_raw = pd.read_csv(csv_file)
df_raw['source_row'] = np.arange(len(df_raw))

print(f'-> Kích thước dữ liệu gốc: {df_raw.shape[0]} hàng, {df_raw.shape[1]} cột.')

# 1. Tìm tất cả các cột ngày định dạng ddThmmyy
time_cols = [col for col in df_raw.columns if re.match(r'^\d{2}Th\d{4}$', col)]
print(f'-> Tìm thấy {len(time_cols)} cột ngày nhị phân.')

# 2. Hàm giải mã tên cột (ví dụ: 19Th0119 -> datetime(2019, 1, 19))
def parse_date_column(col_name):
    day = int(col_name[0:2])
    month = int(col_name[4:6])
    year = int('20' + col_name[6:8])
    return datetime(year, month, day)

# 3. Hàm trích xuất ngày an toàn (lọc bỏ các dòng toàn số 0)
def extract_safe_date(row):
    active_cols = [col for col in time_cols if row[col] == 1]
    if len(active_cols) == 1:
        return parse_date_column(active_cols[0])
    return pd.NaT

df_clean = df_raw.copy()
df_clean['date'] = df_clean.apply(extract_safe_date, axis=1)

# 4. Loại bỏ 22 dòng khuyết ngày để dữ liệu sạch 100%
num_missing = df_clean['date'].isna().sum()
df_clean = df_clean.dropna(subset=['date']).reset_index(drop=True)
print(f'-> Đã loại bỏ {num_missing} dòng khuyết ngày. Dữ liệu sạch: {df_clean.shape[0]} mẫu.')

# 5. Tạo các biến thời gian liên tục và tuần hoàn
df_clean['doy'] = df_clean['date'].dt.dayofyear
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['sin_doy'] = np.sin(2 * np.pi * df_clean['doy'] / 365.25)
df_clean['cos_doy'] = np.cos(2 * np.pi * df_clean['doy'] / 365.25)
# 6. Xóa bỏ 37 cột nhị phân cũ
df_clean = df_clean.drop(columns=time_cols)

# 7. Danh sách các đặc trưng viễn thám và địa hình đầy đủ (không lọc qua BMA)
rs_features = [
    'DEM', 'Lat', 'Long', 'EVI', 'NDBI', 'NDMI', 'NDSI', 'NDVI', 'NDWI', 'VSSI', 'SAVI',
    'COSRI', 'EVI2', 'MSI', 'ND23', 'ND47', 'SI1', 'SI2', 'SI3', 'SI4', 'SI5', 'SI6',
    'SI7', 'SI8', 'SI9', 'CA', 'B', 'G', 'R', 'NIR', 'PCA1', 'PCA2', 'PCA3', 'PCA4', 'PCA5'
]

print(f'-> Bảng dữ liệu hoàn tất chuẩn bị: {df_clean.shape[0]} hàng, {df_clean.shape[1]} cột.')
print(df_clean.columns)

**Cell 4: Phân chia tập dữ liệu chống rò rỉ (Temporal Data Splitting - Chuẩn 70% Train / 30% Test)**
- **Tập Huấn luyện (Train Set):** Toàn bộ năm 2019 đến hết ngày 29/02/2020 (~359 mẫu, chiếm ~70.4% dữ liệu).
- **Tập Kiểm thử (Test Set):** Từ ngày 01/03/2020 đến hết tháng 06/2020 (~151 mẫu, chiếm ~29.6% dữ liệu - giai đoạn cao điểm hạn mặn lịch sử 2020).
- **Đặc trưng mô hình Cây (`X_base` - Theo góp ý của Thầy):**
  + 35 biến viễn thám & địa hình (`rs_features`).
  + 2 biến tuần hoàn `sin_doy`, `cos_doy` để biểu diễn **tính mùa vụ**.
  + 1 biến số ngày tuyệt đối `t` để biểu diễn **thay đổi dài hạn** (long-term trend giữa các năm).
  + Tổng cộng: **38 biến đầu vào**.
- **Đặc trưng Gaussian Process:** Không gian `Lat`, `Long` (đổi sang km, chuẩn hóa theo train) và thời gian kết hợp (`t` tuyệt đối + tuần hoàn `sin_doy, cos_doy`).
- **Outer fold theo ngày:** tạo OOF bằng huấn luyện trên quá khứ và dự báo các ngày phía sau. PSO và trọng số blend được chọn bằng inner fold riêng trong phần train của mỗi outer fold.
- **Nhãn mục tiêu (`y`):** Giữ nguyên nồng độ mặn thực tế `SALINITY_OK` (g/l hoặc ‰).

In [ ]:
%%gp312
target_col = 'SALINITY_OK'
BASE_MODE = 'tree_only'  # 'tree_only' cho các thí nghiệm; 'tree_ridge' cho nền blend.
if BASE_MODE not in {'tree_only', 'tree_ridge'}:
    raise ValueError('BASE_MODE không hợp lệ.')

# ===== CỜ THÍ NGHIỆM =====
# Theo góp ý của Thầy: Dùng đồng thời sin-cos để biểu diễn mùa vụ và số ngày tuyệt đối để biểu diễn thay đổi dài hạn
USE_SEASONAL      = True      # True: dùng đồng thời sin_doy, cos_doy và t cho mô hình nền
RECENCY_HALF_LIFE = None        # None: không đánh trọng số | số ngày (vd 90): hàng càng gần cuối train càng nặng
PSO_CV            = 'forward'   # Nested OOF chỉ dùng các fold tiến theo thời gian.
OOF_FOLDS         = 5
OOF_MIN_SCENES    = 10          # Số ngày quan trắc đầu dùng để khởi tạo, chưa có OOF.
INNER_FOLDS       = 3
INNER_MIN_SCENES  = 4
PSO_SWARMSIZE     = 12
PSO_MAXITER       = 10
MODEL_N_JOBS      = -1
PSO_SEEDS = {'PSO RF': 42, 'PSO Bagging': 43, 'PSO XGB': 44,
             'PSO LightGBM': 45, 'Ridge': 46}
GP_TIME_LS_MAX    = 360.0    # cận trên độ dài tương quan thời gian của GP (ngày)
RUN_TAG = f"seasonal{int(USE_SEASONAL)}_hl{RECENCY_HALF_LIFE}_nested_{PSO_CV}"   # gắn vào tên file kết quả

# 1. Phân chia theo mốc thời gian (nguyên cảnh ảnh, không cắt đôi một ngày)
train_mask = (df_clean['date'] >= '2019-01-01') & (df_clean['date'] < '2020-03-01')
test_mask  = (df_clean['date'] >= '2020-03-01')

# Sắp xếp theo ngày (ổn định) RỒI mới reset index, để mọi mảng bên dưới cùng một thứ tự
df_train = df_clean[train_mask].sort_values('date', kind='stable').reset_index(drop=True)
df_test  = df_clean[test_mask].sort_values('date', kind='stable').reset_index(drop=True)

# Thời gian tuyệt đối: số ngày kể từ cảnh đầu tiên
t_0 = df_train['date'].min()
df_train['t'] = (df_train['date'] - t_0).dt.days
df_test['t']  = (df_test['date']  - t_0).dt.days

# 2. Ma trận đầu vào cho các Mô hình Cây (Viễn thám 35 biến + sin/cos mùa vụ + t dài hạn)
temporal_cols = ['sin_doy', 'cos_doy', 't'] if USE_SEASONAL else []
base_cols = rs_features + temporal_cols
X_train_base = df_train[base_cols].values
X_test_base  = df_test[base_cols].values

# 3. Ma trận (Lat, Long, t) – dùng cho kernel tích tùy chọn ở Cell 6.2
st_cols = ['Lat', 'Long', 't']
X_train_st = df_train[st_cols].values
X_test_st  = df_test[st_cols].values

# 4. Vector biến mục tiêu (Độ mặn thực tế)
y_train = df_train[target_col].values.astype(float)
y_test  = df_test[target_col].values.astype(float)

if df_train.empty or df_test.empty:
    raise ValueError('Train hoặc test rỗng.')
for frame in [df_train, df_test]:
    if not np.isfinite(frame[base_cols + [target_col]].to_numpy(dtype=float)).all():
        raise ValueError('Dữ liệu đầu vào có NaN hoặc vô cực.')

# 5. Outer folds tạo OOF; inner folds được tạo riêng trong train của mỗi outer fold.
def forward_splits(dates, n_folds=5, min_scenes=10):
    """Chia theo ngày quan trắc; train luôn đứng hoàn toàn trước validation."""
    dates = np.asarray(dates, dtype='datetime64[ns]')
    if dates.ndim != 1 or np.isnat(dates).any():
        raise ValueError('dates phải là vector ngày hợp lệ, không có NaT.')
    unique_dates = np.unique(dates)
    if n_folds < 1 or min_scenes < 1 or len(unique_dates) <= min_scenes:
        raise ValueError('Không đủ ngày quan trắc cho forward CV; kiểm tra min_scenes.')
    n_folds = min(n_folds, len(unique_dates) - min_scenes)
    edges = np.linspace(min_scenes, len(unique_dates), n_folds + 1).astype(int)
    result = []
    for a, b in zip(edges[:-1], edges[1:]):
        tr = np.flatnonzero(dates < unique_dates[a])
        va = np.flatnonzero(np.isin(dates, unique_dates[a:b]))
        assert len(tr) and len(va) and dates[tr].max() < dates[va].min()
        result.append((tr, va))
    return result


if PSO_CV != 'forward':
    raise ValueError("Nested temporal OOF yêu cầu PSO_CV='forward'.")
cv_strategy = forward_splits(df_train['date'].values, OOF_FOLDS, OOF_MIN_SCENES)
splits = cv_strategy  # Tương thích các cell tùy chọn phía sau.

def recency_weights(t_days, t_ref, half_life, floor=0.05):
    if half_life is None:
        return np.ones(len(t_days))
    return np.maximum(0.5 ** ((t_ref - np.asarray(t_days, float)) / half_life), floor)

def fit_weighted(model, X, y, w):
    m = clone(model)
    if isinstance(m, Pipeline):
        m.fit(X, y, **{m.steps[-1][0] + '__sample_weight': w})
    else:
        m.fit(X, y, sample_weight=w)
    return m

w_train = recency_weights(df_train['t'].values, df_train['t'].max(), RECENCY_HALF_LIFE)

# 6. In thông số kiểm tra
print('=' * 80)
print('THÔNG TIN PHÂN CHIA TẬP DỮ LIỆU HUẤN LUYỆN & KIỂM THỬ (TEMPORAL SPLIT)')
print('=' * 80)
print(f'-> Tập Train (2019 đến hết T2/2020): {X_train_base.shape[0]} mẫu, {df_train["date"].nunique()} cảnh ({X_train_base.shape[0]/df_clean.shape[0]*100:.1f}%) | Độ mặn TB: {y_train.mean():.2f} ‰')
print(f'-> Tập Test  (T3 đến T6/2020):        {X_test_base.shape[0]} mẫu, {df_test["date"].nunique()} cảnh ({X_test_base.shape[0]/df_clean.shape[0]*100:.1f}%) | Độ mặn TB: {y_test.mean():.2f} ‰')
print(f'-> Số đặc trưng đưa vào mô hình Cây:  {X_train_base.shape[1]} biến (35 viễn thám & địa hình + {len(temporal_cols)} biến thời gian: {temporal_cols})')
print('Nền GP:', BASE_MODE)
print(f'-> Cờ: USE_SEASONAL={USE_SEASONAL} | RECENCY_HALF_LIFE={RECENCY_HALF_LIFE} | PSO_CV={PSO_CV} | GP_TIME_LS_MAX={GP_TIME_LS_MAX}')
print(f'-> Thang đo mặn thực tế: min={y_train.min():.2f}, mean={y_train.mean():.2f}, max={y_train.max():.2f} (‰)')

**Cell 4.1: Kiểm tra nhanh dịch chuyển phân phối và trạm lặp lại (chạy trước khi đọc kết quả mô hình)**
- Mùa test (T3–T6/2020) có mặn hơn hẳn mùa train hay không (so sánh trung bình, p95, tỉ lệ mẫu vượt `max(y_train)`).
- Các trạm test có xuất hiện trong train không (quyết định bạn đang đo "dự báo tương lai tại trạm đã biết" hay "dự báo tại trạm mới").

In [ ]:
%%gp312
print('--- Phân phối độ mặn ---')
print(f'Train: mean={y_train.mean():.2f}  p95={np.percentile(y_train,95):.2f}  max={y_train.max():.2f}')
print(f'Test : mean={y_test.mean():.2f}  p95={np.percentile(y_test,95):.2f}  max={y_test.max():.2f}')
print(f'Tỉ lệ mẫu test vượt max(y_train): {(y_test > y_train.max()).mean():.3f}')

print('\n--- Số mẫu theo cảnh ---')
print('Train:', df_train.groupby('date').size().describe()[['count','min','max']].astype(int).to_dict())
print(df_test.groupby('date').size().to_string())

print('\n--- Trạm lặp lại giữa train và test ---')
_key = lambda d: set(zip(d['Lat'].round(5), d['Long'].round(5)))
tr_pts, te_pts = _key(df_train), _key(df_test)
print(f'Số trạm train: {len(tr_pts)} | test: {len(te_pts)} | trạm test đã có trong train: {len(tr_pts & te_pts)}/{len(te_pts)}')

**Cell 5: Khởi tạo các mô hình cơ sở (Decision Tree, Bagging, Random Forest)**
- 1. Decision Tree Regressor (Cây quyết định mặc định)
- 2. Bagging Regressor cơ sở theo Table 5 bài báo (`n_estimators = 10`)
- 3. Random Forest cơ sở theo Table 5 bài báo (`n_estimators = 100`)
- (1D-CNN được chuyển xuống Cell 7.1 và tắt mặc định, vì tích chập trên trục đặc trưng của dữ liệu bảng không có ý nghĩa.)

In [ ]:
%%gp312
# 1. Decision Tree cơ sở
dt_base = DecisionTreeRegressor(max_depth=7, min_samples_leaf=5, random_state=42)

# 2. Bagging Regressor cơ sở theo bài báo (Table 5)
bagging_base = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=7, min_samples_leaf=5, random_state=42),
    n_estimators=10,
    random_state=42,
    n_jobs=-1
)

# 3. Random Forest cơ sở theo bài báo (Table 5)
rf_base = RandomForestRegressor(
    n_estimators=100,
    max_depth=7,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

print('Đã khởi tạo xong các mô hình cơ sở: Decision Tree, Bagging, Random Forest.')

**Cell 6: Định nghĩa mô hình và chọn siêu tham số PSO trong từng outer fold**
- Giữ các khoảng tìm kiếm PSO hiện có. Mỗi outer fold chọn tham số bằng các inner fold chỉ thuộc phần quá khứ của fold đó.
- `INNER_FOLDS=3`, `INNER_MIN_SCENES=4`; ngân sách mỗi lần PSO là `PSO_SWARMSIZE=12`, `PSO_MAXITER=10`.
- Decision Tree, Bagging và Random Forest cơ sở giữ tham số cố định; Ridge và các mô hình PSO được tối ưu riêng từng fold.
- Các cell dưới đây chỉ định nghĩa hàm. PSO bắt đầu chạy ở Cell 7, trước hết tạo nested OOF, sau đó chọn tham số mô hình cuối trên toàn bộ train.

In [ ]:
%%gp312
# Factory dùng chung cho tuning, OOF và fit cuối để cùng một bộ tham số tạo cùng mô hình.
PSO_SEARCH = {
    'PSO RF': {
        'names': ['n_estimators', 'max_depth', 'min_samples_leaf', 'max_features', 'min_samples_split'],
        'lower': [120, 3, 3, 0.5, 8], 'upper': [250, 8, 10, 0.8, 20],
    },
    'PSO Bagging': {
        'names': ['n_estimators', 'max_depth', 'min_samples_leaf', 'max_features', 'min_samples_split'],
        'lower': [120, 3, 3, 0.5, 8], 'upper': [250, 8, 10, 0.8, 20],
    },
    'PSO XGB': {
        'names': ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree',
                  'min_child_weight', 'gamma', 'reg_lambda', 'reg_alpha'],
        'lower': [50, 2, 0.01, 0.6, 0.5, 3, 0.0, 1.0, 0.0],
        'upper': [100, 4, 0.08, 0.85, 0.8, 15, 5.0, 15.0, 5.0],
    },
    'PSO LightGBM': {
        'names': ['n_estimators', 'num_leaves', 'learning_rate', 'subsample', 'colsample_bytree',
                  'min_child_samples', 'reg_lambda', 'reg_alpha'],
        'lower': [50, 4, 0.01, 0.6, 0.5, 15, 2.0, 1.0],
        'upper': [100, 8, 0.06, 0.8, 0.7, 25, 8.0, 3.0],
    },
    'Ridge': {'names': ['alpha'], 'lower': [0.1], 'upper': [1000.0]},
}
INTEGER_PARAMETERS = {'n_estimators', 'max_depth', 'min_samples_leaf',
                      'min_samples_split', 'num_leaves', 'min_child_samples'}
MODEL_NAMES = ['Decision Tree', 'Bagging', 'PSO Bagging', 'Random Forest',
               'PSO RF', 'PSO XGB', 'PSO LightGBM']


def decode_parameters(name, values):
    return {key: int(round(value)) if key in INTEGER_PARAMETERS else float(value)
            for key, value in zip(PSO_SEARCH[name]['names'], values)}


def build_base_model(name, parameters=None):
    parameters = dict(parameters or {})
    fixed_models = {'Decision Tree': dt_base, 'Bagging': bagging_base, 'Random Forest': rf_base}
    if name in fixed_models:
        model = clone(fixed_models[name])
        if 'n_jobs' in model.get_params():
            model.set_params(n_jobs=MODEL_N_JOBS)
        return model
    if name == 'Ridge':
        return make_pipeline(StandardScaler(), Ridge(random_state=42, **parameters))
    if name == 'PSO RF':
        return RandomForestRegressor(random_state=42, n_jobs=MODEL_N_JOBS, **parameters)
    if name == 'PSO Bagging':
        tree_parameters = {key: parameters.pop(key)
                           for key in ['max_depth', 'min_samples_leaf', 'min_samples_split']}
        tree = DecisionTreeRegressor(random_state=42, **tree_parameters)
        return BaggingRegressor(estimator=tree, bootstrap=True, random_state=42,
                                n_jobs=MODEL_N_JOBS, **parameters)
    if name == 'PSO XGB':
        return XGBRegressor(random_state=42, n_jobs=MODEL_N_JOBS, **parameters)
    if name == 'PSO LightGBM':
        return LGBMRegressor(max_depth=-1, subsample_freq=1, random_state=42,
                             verbose=-1, n_jobs=MODEL_N_JOBS, **parameters)
    raise ValueError('Tên mô hình không hợp lệ: ' + name)


def fit_dated_model(model, X, y, dates):
    """Fit cả scaler/Ridge trong phạm vi dữ liệu được truyền vào; recency lấy mốc cuối cục bộ."""
    dates = np.asarray(dates, dtype='datetime64[ns]')
    days = (dates - dates.min()) / np.timedelta64(1, 'D')
    weights = recency_weights(days, days.max(), RECENCY_HALF_LIFE)
    return fit_weighted(model, X, y, weights)

### Hàm chọn tham số trong lịch sử từng fold
PSO chỉ nhận dữ liệu của phần train được truyền vào. Log lưu cả chỉ số inner fold để kiểm tra. Trong chế độ blend, inner OOF chọn trọng số hoàn toàn trong lịch sử outer fold.

In [ ]:
%%gp312
import inspect


def extract_pso_result(result):
    if hasattr(result, 'x') and hasattr(result, 'fun'):
        return np.asarray(result.x, dtype=float), float(result.fun)
    if isinstance(result, dict) and 'x' in result and 'fun' in result:
        return np.asarray(result['x'], dtype=float), float(result['fun'])
    if isinstance(result, (tuple, list)) and len(result) >= 2:
        return np.asarray(result[0], dtype=float), float(result[1])
    raise TypeError('Không nhận diện được kết quả PSO.')


def run_seeded_pso(objective, name):
    space = PSO_SEARCH[name]
    options = {'swarmsize': PSO_SWARMSIZE, 'maxiter': PSO_MAXITER}
    if 'seed' in inspect.signature(pso).parameters:
        return pso(objective, space['lower'], space['upper'], seed=PSO_SEEDS[name], **options)
    # Tương thích pyswarm cũ, không để seed của một lần tuning ảnh hưởng lần khác.
    random_state = np.random.get_state()
    try:
        np.random.seed(PSO_SEEDS[name])
        return pso(objective, space['lower'], space['upper'], **options)
    finally:
        np.random.set_state(random_state)


def tune_on_history(name, X, y, dates, source_indices, context):
    """Chỉ nhận lịch sử của fold đang xét, không nhận validation ngoài hay test."""
    if name not in PSO_SEARCH:
        return {}, None
    inner_splits = forward_splits(dates, INNER_FOLDS, INNER_MIN_SCENES)

    def objective(values):
        parameters = decode_parameters(name, values)
        errors = []
        for tr, va in inner_splits:
            model = fit_dated_model(build_base_model(name, parameters), X[tr], y[tr], dates[tr])
            errors.append(np.mean((y[va] - model.predict(X[va])) ** 2))
        return float(np.mean(errors))

    values, best_mse = extract_pso_result(run_seeded_pso(objective, name))
    parameters = decode_parameters(name, values)
    tuning_log.append({
        'context': context, 'model': name, 'N': len(y),
        'fit_start': str(dates.min()), 'fit_end': str(dates.max()),
        'source_indices': np.asarray(source_indices).tolist(),
        'inner_cv_indices': [
            {'train_indices': np.asarray(source_indices)[tr].tolist(),
             'validation_indices': np.asarray(source_indices)[va].tolist()}
            for tr, va in inner_splits],
        'parameters': parameters.copy(), 'xopt': values.tolist(),
        'best_inner_cv_mse': best_mse, 'seed': PSO_SEEDS[name],
    })
    return parameters, best_mse


def predict_inner_oof(name, parameters, X, y, dates):
    """Các dự báo bên trong lịch sử outer fold dùng để chọn trọng số blend."""
    prediction = np.full(len(y), np.nan)
    for tr, va in forward_splits(dates, INNER_FOLDS, INNER_MIN_SCENES):
        model = fit_dated_model(build_base_model(name, parameters), X[tr], y[tr], dates[tr])
        prediction[va] = model.predict(X[va])
    return prediction


print('Đã định nghĩa PSO cục bộ; Cell 7 sẽ chọn tham số riêng trong từng outer fold.')

**Cell 6.1: Gaussian Process học phần sai số (residual) theo Không gian - Thời gian**

Theo đúng góp ý của Thầy:
- **Tạo residual bằng dự đoán OOF chỉ trên tập train:** $r_{train} = y_{train} - \hat{y}_{train, OOF}$, tuyệt đối không bị rò rỉ dữ liệu test.
- **Kernel thời gian:** Kết hợp **Matern 3/2 trên số ngày tuyệt đối $t$** (nắm bắt xu hướng thay đổi dài hạn) cộng với **Kernel tuần hoàn trên chu kỳ ngày trong năm** (`sin_doy, cos_doy`) để nắm bắt tính mùa vụ.
- **Kernel không gian:** Dùng **Matern 3/2 (hoặc RBF) trên tọa độ trạm $(x, y)$** (theo km chuẩn hóa) để nắm bắt dị thường địa phương giữa các vùng cửa sông và nội đồng.
- **Dự báo hiệu chỉnh (Static Test Holdout):** Gaussian Process được fit hoàn toàn trên tập train và dự báo phần bù $\hat{r}_{test}$ cho tập kiểm thử mà không cần dùng đến nhãn test:
  $$\hat{y}_{test, hybrid} = \hat{y}_{test, base} + \hat{r}_{test, GP}$$


## Huấn luyện nested OOF và mô hình cuối — chạy nặng trên Kaggle

Mỗi outer fold chạy PSO trên inner CV của phần quá khứ, rồi fit trên toàn bộ quá khứ và dự báo outer validation. `oof_tree` lưu dự báo cây thô; `oof_blend` lưu dự báo nền sau clip (bằng cây thuần khi `BASE_MODE='tree_only'`).

Sau khi khóa OOF, chọn lại tham số cây cuối bằng CV trong toàn bộ train, fit cây cuối và fit GP trên residual OOF. Nhãn test chỉ được dùng chấm điểm. Ridge được giữ để đối chiếu và hỗ trợ tùy chọn blend; trong chế độ cây thuần, trọng số luôn là `(1, 0)`.

`Base_CV_*` chỉ chấm nền trước GP. Chưa có bước outer CV cho toàn bộ mô hình cây + GP.

In [ ]:
%%gp312
# ==============================================================================
# ĐỊNH NGHĨA GAUSSIAN PROCESS MÙA VỤ + KHÔNG GIAN + THỜI GIAN TUYỆT ĐỐI
# Theo đúng góp ý của Thầy: Matern(t) + Tuần hoàn sin/cos/doy + Không gian Matern/RBF
# ==============================================================================
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel as C, WhiteKernel

# 1. Trích xuất các ma trận đầu vào cho GP
lat0, lon0 = df_train['Lat'].mean(), df_train['Long'].mean()

def get_xy_coords(d):
    x = (d['Long'].values - lon0) * 111.32 * np.cos(np.radians(lat0))   # km
    y = (d['Lat'].values  - lat0) * 110.57                                # km
    return np.c_[x, y]

xy_tr = get_xy_coords(df_train)
xy_te = get_xy_coords(df_test)
xy_sd = xy_tr.std(axis=0)
if not np.isfinite(xy_sd).all() or (xy_sd <= 0).any():
    raise ValueError('Không thể chuẩn hóa tọa độ train.')
xy_tr /= xy_sd
xy_te /= xy_sd

# Thời gian tuyệt đối (ngày)
t_tr = df_train['t'].values.astype(float)[:, None]
t_te = df_test['t'].values.astype(float)[:, None]

# Tọa độ vòng tròn mùa vụ (sin_doy, cos_doy)
sc_tr = df_train[['sin_doy', 'cos_doy']].values.astype(float)
sc_te = df_test[['sin_doy', 'cos_doy']].values.astype(float)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Kernel, Hyperparameter, RBF, Matern, ConstantKernel as C, WhiteKernel

class ActiveDims(Kernel):
    """Bọc một kernel để nó chỉ nhìn vào một số cột của X."""
    def __init__(self, kernel, dims):
        self.kernel = kernel
        self.dims = dims
    def get_params(self, deep=True):
        params = dict(kernel=self.kernel, dims=self.dims)
        if deep:
            params.update(('kernel__' + k, v) for k, v in self.kernel.get_params().items())
        return params
    @property
    def hyperparameters(self):
        return [Hyperparameter('kernel__' + h.name, h.value_type, h.bounds, h.n_elements)
                for h in self.kernel.hyperparameters]
    @property
    def theta(self):
        return self.kernel.theta
    @theta.setter
    def theta(self, theta):
        self.kernel.theta = theta
    @property
    def bounds(self):
        return self.kernel.bounds
    def __call__(self, X, Y=None, eval_gradient=False):
        d = list(self.dims)
        Xs = np.atleast_2d(X)[:, d]
        Ys = None if Y is None else np.atleast_2d(Y)[:, d]
        return self.kernel(Xs, Ys, eval_gradient=eval_gradient)
    def diag(self, X):
        return self.kernel.diag(np.atleast_2d(X)[:, list(self.dims)])
    def is_stationary(self):
        return self.kernel.is_stationary()

def build_joint_kernel(var_r):
    a = max(var_r / 3.0, 0.05)
    k_time  = ActiveDims(C(a, (1e-2, 1e3)) * Matern(min(45.0, GP_TIME_LS_MAX), (15.0, GP_TIME_LS_MAX), nu=1.5), dims=[0])
    k_seas  = ActiveDims(C(a, (1e-2, 1e3)) * RBF([1.0, 1.0], (0.2, 5.0)), dims=[1, 2])
    k_space = ActiveDims(C(a, (1e-2, 1e3)) * Matern([1.0, 1.0], (0.1, 20.0), nu=1.5), dims=[3, 4])
    return k_time + k_seas + k_space + WhiteKernel(a, (1e-2, 1e3))

def fit_correction_joint(t, sc, xy, r, n_restarts=3, seed=42):
    X = np.c_[t, sc, xy]          # cột 0 = t | 1-2 = sin, cos của chu kỳ năm | 3-4 = tọa độ km chuẩn hóa
    gp = GaussianProcessRegressor(build_joint_kernel(np.var(r)), n_restarts_optimizer=n_restarts, random_state=seed)
    return gp.fit(X, r)

def predict_correction_joint(gp, t, sc, xy):
    return gp.predict(np.c_[t, sc, xy])

def show_hparams(gp):
    th = np.exp(gp.kernel_.theta); i = 0
    for h in gp.kernel_.hyperparameters:
        n = h.n_elements; val = th[i:i+n]; i += n
        lo, hi = h.bounds[:, 0], h.bounds[:, 1]
        edge = np.isclose(val, lo, rtol=0.02) | np.isclose(val, hi, rtol=0.02)
        short = h.name.replace('kernel__', '').replace('k1__', '').replace('k2__', '')
        print(f'{short:34s} {np.round(val, 3)}' + ('   <-- DÍNH CẬN' if edge.any() else ''))

### Tạo outer OOF, fit cây cuối và GP
Cell này thực hiện các bước huấn luyện vừa mô tả, sau đó hiển thị điểm mô hình nền và điểm test. Không dùng nhãn test để tối ưu.

In [ ]:
%%gp312
# Tạo OOF trước khi chọn/fit các mô hình cuối trên toàn bộ train.
oof_splits = list(cv_strategy)
tuning_log, oof_fold_log = [], []
BLEND_METHOD = 'stacking'


def evaluate_oof(y_true, oof_pred):
    """Chỉ chấm các dòng có OOF hợp lệ; không thay warm-up bằng 0."""
    y_true = np.asarray(y_true, dtype=float)
    oof_pred = np.asarray(oof_pred, dtype=float)
    if y_true.shape != oof_pred.shape:
        raise ValueError('Nhãn và dự báo OOF phải có cùng kích thước.')
    valid = np.isfinite(y_true) & np.isfinite(oof_pred)
    if not valid.any():
        raise ValueError('Không có OOF hợp lệ; kiểm tra cv_strategy.')
    metrics = evaluate_model(y_true[valid], oof_pred[valid])
    return {**metrics, 'N': int(valid.sum())}


def get_blend_weights(oof_tree, oof_lin, y_true, method='stacking'):
    """Fit trọng số trên OOF hợp lệ; tổng hai trọng số bằng 1."""
    if BASE_MODE == 'tree_only':
        return 1.0, 0.0
    from scipy.optimize import minimize_scalar

    oof_tree = np.asarray(oof_tree, dtype=float)
    oof_lin = np.asarray(oof_lin, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    if not (oof_tree.shape == oof_lin.shape == y_true.shape):
        raise ValueError('Các đầu vào blend phải có cùng kích thước.')
    valid = np.isfinite(oof_tree) & np.isfinite(oof_lin) & np.isfinite(y_true)
    if not valid.any():
        raise ValueError('Không có OOF hợp lệ để fit trọng số blend.')
    tree, linear, target = oof_tree[valid], oof_lin[valid], y_true[valid]

    if method == 'stacking':
        # Cùng công thức cho OOF/train/test: blend trước, clip sau.
        loss = lambda w: np.mean(
            (target - np.clip(w * tree + (1 - w) * linear, 0, None)) ** 2
        )
        candidate = float(minimize_scalar(
            loss, bounds=(0.0, 1.0), method='bounded'
        ).x)
        # Cho phép chọn đúng cây thuần hoặc Ridge thuần.
        w_tree = float(min((0.0, candidate, 1.0), key=loss))
    elif method == 'inv_mse':
        errors = np.array([
            mean_squared_error(target, np.clip(tree, 0, None)),
            mean_squared_error(target, np.clip(linear, 0, None)),
        ])
        inv = 1.0 / np.maximum(errors, 1e-12)
        w_tree = float(inv[0] / inv.sum())
    else:
        raise ValueError("method phải là 'stacking' hoặc 'inv_mse'")
    return w_tree, 1.0 - w_tree


def make_nested_oof(X, y, dates, model_names):
    dates = np.asarray(dates, dtype='datetime64[ns]')
    y = np.asarray(y)
    outer_splits = forward_splits(dates, OOF_FOLDS, OOF_MIN_SCENES)
    ridge_oof = np.full(len(y), np.nan)
    fold_id = np.full(len(y), -1, dtype=int)
    states = {name: {
        'oof_tree': np.full(len(y), np.nan),
        'oof_blend': np.full(len(y), np.nan),
        'oof_w_tree': np.full(len(y), np.nan),
    } for name in model_names}

    for fold, (tr, va) in enumerate(outer_splits, start=1):
        assert dates[tr].max() < dates[va].min()
        assert np.all(fold_id[va] == -1)
        fold_id[va] = fold
        X_history, y_history, dates_history = X[tr], y[tr], dates[tr]
        context = f'outer_fold_{fold}'
        print(f'{context}: train={len(tr)}, OOF={len(va)}; '
              f'{dates[tr].max()} < {dates[va].min()}', flush=True)

        ridge_parameters, _ = tune_on_history(
            'Ridge', X_history, y_history, dates_history, tr, context)
        ridge_inner = predict_inner_oof(
            'Ridge', ridge_parameters, X_history, y_history, dates_history)
        ridge = fit_dated_model(build_base_model('Ridge', ridge_parameters),
                                X_history, y_history, dates_history)
        ridge_oof[va] = ridge.predict(X[va])

        for name in model_names:
            parameters, _ = tune_on_history(
                name, X_history, y_history, dates_history, tr, context)
            tree_inner = predict_inner_oof(
                name, parameters, X_history, y_history, dates_history)
            w_tree, w_lin = get_blend_weights(
                tree_inner, ridge_inner, y_history, method=BLEND_METHOD)
            tree = fit_dated_model(build_base_model(name, parameters),
                                   X_history, y_history, dates_history)
            tree_prediction = tree.predict(X[va])
            states[name]['oof_tree'][va] = tree_prediction
            states[name]['oof_blend'][va] = np.clip(
                w_tree * tree_prediction + w_lin * ridge_oof[va], 0, None)
            states[name]['oof_w_tree'][va] = w_tree
            oof_fold_log.append({
                'fold': fold, 'model': name, 'train_indices': tr.tolist(),
                'validation_indices': va.tolist(),
                'fit_end': str(dates[tr].max()), 'prediction_start': str(dates[va].min()),
                'prediction_end': str(dates[va].max()),
                'tree_parameters': parameters.copy(), 'ridge_parameters': ridge_parameters.copy(),
                'blend_weights': [w_tree, w_lin],
            })
            print(f'  {name}: w_tree={w_tree:.3f}', flush=True)
    return ridge_oof, states, fold_id


def row(name, ev_tr, ev_cv, ev_te):
    # Với dòng + GP, hai cột Base vẫn chấm nền trước khi thêm GP.
    return {
        'Model': name,
        'Base_Train_R2 (in-sample)': ev_tr['R2'],
        'Base_CV_R2 (OOF)': ev_cv['R2'],
        'Base_CV_RMSE (‰)': ev_cv['RMSE (‰)'],
        'Base_CV_N': ev_cv.get('N', np.nan),
        'Test_RMSE (‰)': ev_te['RMSE (‰)'],
        'Test_MAE (‰)': ev_te['MAE (‰)'],
        'Test_R2': ev_te['R2'],
        'Test_Ex_VAR (%)': ev_te['Ex_VAR (%)'],
        'Test_Bias (‰)': ev_te['Bias (‰)'],
    }


oof_linear, nested_oof, oof_fold_id = make_nested_oof(
    X_train_base, y_train, df_train['date'].values, MODEL_NAMES)

# Sau khi khóa OOF, chọn tham số cuối trong toàn bộ train để dự báo test.
final_parameters = {}
for name in ['Ridge'] + MODEL_NAMES:
    print(f'Chọn tham số mô hình cuối: {name}', flush=True)
    final_parameters[name], _ = tune_on_history(
        name, X_train_base, y_train, df_train['date'].values,
        np.arange(len(y_train)), 'final_train')

models_tree = {name: build_base_model(name, final_parameters[name]) for name in MODEL_NAMES}
pso_linear_rigde = build_base_model('Ridge', final_parameters['Ridge'])
pso_rf_model, pso_bagging_model = models_tree['PSO RF'], models_tree['PSO Bagging']
pso_xgb_model, pso_lgbm_model = models_tree['PSO XGB'], models_tree['PSO LightGBM']
rigde_full = fit_dated_model(pso_linear_rigde, X_train_base, y_train, df_train['date'].values)
pred_linear_train = rigde_full.predict(X_train_base)
pred_linear_test = rigde_full.predict(X_test_base)

results, predictions, store = [], {}, {}
oof_mean = np.full(len(y_train), np.nan)
for tr, va in oof_splits:
    oof_mean[va] = y_train[tr].mean()
results.append(row(
    'Baseline: trung bình train',
    evaluate_model(y_train, np.full(len(y_train), y_train.mean())),
    evaluate_oof(y_train, oof_mean),
    evaluate_model(y_test, np.full(len(y_test), y_train.mean())),
))
results.append(row(
    'Baseline: Ridge đơn lẻ',
    evaluate_model(y_train, np.clip(pred_linear_train, 0, None)),
    evaluate_oof(y_train, np.clip(oof_linear, 0, None)),
    evaluate_model(y_test, np.clip(pred_linear_test, 0, None)),
))

for name, model in models_tree.items():
    print(f'Fit mô hình cuối và GP: {name}', flush=True)
    oof_tree = nested_oof[name]['oof_tree']
    oof_blend = nested_oof[name]['oof_blend']
    valid = np.isfinite(oof_tree) & np.isfinite(oof_linear) & np.isfinite(oof_blend)
    if not valid.any():
        raise ValueError(f'{name}: không có nested OOF hợp lệ.')

    # Trọng số này CHỈ dùng cho mô hình cuối; không tính lại dự báo outer OOF.
    w_tree, w_lin = get_blend_weights(oof_tree, oof_linear, y_train, method=BLEND_METHOD)
    tree_full = fit_dated_model(model, X_train_base, y_train, df_train['date'].values)
    pred_train_tree_raw = tree_full.predict(X_train_base)
    pred_test_tree_raw = tree_full.predict(X_test_base)
    pred_test_tree_only = np.clip(pred_test_tree_raw, 0, None)
    pred_train_base = np.clip(w_tree * pred_train_tree_raw + w_lin * pred_linear_train, 0, None)
    pred_test_base = np.clip(w_tree * pred_test_tree_raw + w_lin * pred_linear_test, 0, None)

    ev_tr = evaluate_model(y_train, pred_train_base)
    ev_cv = evaluate_oof(y_train, oof_blend)
    ev_te_tree = evaluate_model(y_test, pred_test_tree_only)
    ev_te_base = evaluate_model(y_test, pred_test_base)
    base_label = name if BASE_MODE == 'tree_only' else f'{name} + Ridge'
    results.append(row(base_label, ev_tr, ev_cv, ev_te_base))
    if name == 'PSO RF':
        results.append(row(
            f'{name} (thuần gốc)',
            evaluate_model(y_train, np.clip(pred_train_tree_raw, 0, None)),
            evaluate_oof(y_train, np.clip(oof_tree, 0, None)), ev_te_tree,
        ))

    residual_oof = y_train - oof_blend
    gp = fit_correction_joint(t_tr[valid], sc_tr[valid], xy_tr[valid], residual_oof[valid])
    corr_test = predict_correction_joint(gp, t_te, sc_te, xy_te)
    show_hparams(gp)
    pred_test_hybrid = np.clip(pred_test_base + corr_test, 0, None)
    results.append(row(
        f'{name} + GP (Đề xuất)', ev_tr, ev_cv, evaluate_model(y_test, pred_test_hybrid)))
    predictions[name] = {'tree_only': pred_test_tree_only,
                         'base': pred_test_base, 'hybrid': pred_test_hybrid}
    store[name] = {
        'tree': tree_full, 'gp': gp, 'pred_test_base': pred_test_base,
        'residual_oof': residual_oof, 'oof_mask': valid.copy(),
        'oof_indices': np.flatnonzero(valid), 'blend_weights': (w_tree, w_lin),
        'oof_tree': oof_tree.copy(), 'oof_ridge': oof_linear.copy(),
        'oof_blend': oof_blend.copy(),
        'residual_tree_oof': y_train - np.clip(oof_tree, 0, None),
        'oof_w_tree': nested_oof[name]['oof_w_tree'].copy(),
        'oof_fold_id': oof_fold_id.copy(),
        'gp_residual_kind': BASE_MODE, 'oof_method': 'nested_forward',
    }

results_df = pd.DataFrame(results).set_index('Model')
print('\n' + '=' * 95)
print('GP học phần dư nested OOF trong train; PSO cục bộ từng fold, nền theo BASE_MODE.')
print('Base_CV chấm nền cây + Ridge trước GP; chưa phải CV của toàn bộ mô hình có GP.')
print('=' * 95)
display(results_df)

## Kiểm tra trước khi lưu
Đối chiếu thứ tự mẫu, tính tách biệt theo thời gian của inner/outer fold, các hàng warm-up, residual GP và cấu trúc kernel cộng cơ sở.

In [ ]:
%%gp312
# Chặn việc xuất bộ model nếu lệch fold, thứ tự mẫu hoặc residual.
from sklearn.gaussian_process.kernels import Sum

def additive_terms(kernel):
    if isinstance(kernel, Sum):
        return additive_terms(kernel.k1) + additive_terms(kernel.k2)
    return [kernel]

fold_rows = []
coverage = np.zeros(len(y_train), dtype=int)
for fold, (tr, va) in enumerate(oof_splits, 1):
    assert not np.intersect1d(tr, va).size
    assert df_train.date.iloc[tr].max() < df_train.date.iloc[va].min()
    coverage[va] += 1
    fold_rows.append({'fold': fold, 'n_train': len(tr), 'n_oof': len(va),
                      'train_end': df_train.date.iloc[tr].max(),
                      'oof_start': df_train.date.iloc[va].min(),
                      'oof_end': df_train.date.iloc[va].max()})
assert coverage.max() == 1
for entry in tuning_log:
    allowed = np.asarray(entry['source_indices'])
    for inner in entry['inner_cv_indices']:
        tr, va = np.asarray(inner['train_indices']), np.asarray(inner['validation_indices'])
        assert np.isin(tr, allowed).all() and np.isin(va, allowed).all()
        assert df_train.date.iloc[tr].max() < df_train.date.iloc[va].min()
    if entry['context'].startswith('outer_fold_'):
        fold = int(entry['context'].rsplit('_', 1)[1])
        np.testing.assert_array_equal(allowed, oof_splits[fold - 1][0])
G_train_export = np.c_[t_tr, sc_tr, xy_tr]
for name, state in store.items():
    mask = state['oof_mask']
    np.testing.assert_array_equal(mask, coverage == 1)
    assert np.isnan(state['oof_tree'][~mask]).all()
    assert np.isnan(state['residual_oof'][~mask]).all()
    np.testing.assert_allclose(state['residual_oof'][mask],
                               y_train[mask] - state['oof_blend'][mask])
    np.testing.assert_allclose(state['gp'].X_train_, G_train_export[mask])
    np.testing.assert_allclose(state['gp'].y_train_, state['residual_oof'][mask])
    if BASE_MODE == 'tree_only':
        np.testing.assert_allclose(state['residual_oof'][mask], state['residual_tree_oof'][mask])
    terms = additive_terms(state['gp'].kernel_)
    assert len(terms) == 4
    assert sorted(tuple(term.dims) for term in terms if hasattr(term, 'dims')) == [(0,), (1, 2), (3, 4)]
    assert any(isinstance(term, WhiteKernel) for term in terms)

display(pd.DataFrame(fold_rows))
print(f'Kiểm tra đạt: {int((coverage == 1).sum())} OOF; {int((coverage == 0).sum())} warm-up mỗi mô hình.')
print('GP cơ sở chưa có tương tác; nguồn OOF khi nạp lại dự kiến: saved_oof_tree.')


## Xuất bộ model
Lưu cây/GP đã fit, OOF cây thô, residual, mặt nạ và fold ID vào PKL; JSON lưu cấu hình, phiên bản môi trường, nguồn và log tuning. Xuất thêm CSV OOF và ZIP tải về; không ghi đè bộ cũ.

In [ ]:
%%gp312
# Chạy cell này ngay sau mỗi lần huấn luyện; mỗi lần lưu một thư mục riêng.
import hashlib
import importlib.metadata as metadata
import json
import shutil
import sys
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd

required = ['models_tree', 'store', 'results_df', 'predictions', 'base_cols']
missing = [key for key in required if key not in globals()]
if missing:
    raise RuntimeError(f'Chạy cell huấn luyện trước khi lưu. Thiếu: {missing}')


def json_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return value if np.isfinite(value) else str(value)
    if isinstance(value, np.ndarray):
        return json_value(value.tolist())
    if isinstance(value, dict):
        return {str(key): json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_value(item) for item in value]
    return str(value)


now = datetime.now(timezone(timedelta(hours=7)))
default_root = '/kaggle/working/training_runs' if Path('/kaggle/working').is_dir() else 'training_runs'
save_root = Path(globals().get('TRAINING_SAVE_ROOT', default_root))
run_dir = save_root / f'{now:%Y%m%d_%H%M%S_%f}_{uuid.uuid4().hex[:8]}'
run_dir.mkdir(parents=True, exist_ok=False)

config_keys = [
    'BASE_MODE', 'RUN_TAG', 'USE_SEASONAL', 'RECENCY_HALF_LIFE', 'PSO_CV',
    'GP_TIME_LS_MAX', 'BLEND_METHOD', 'OOF_FOLDS', 'OOF_MIN_SCENES',
    'INNER_FOLDS', 'INNER_MIN_SCENES', 'PSO_SWARMSIZE', 'PSO_MAXITER', 'MODEL_N_JOBS',
    'PSO_SEEDS', 'target_col',
]
preprocessing = {key: globals()[key] for key in ['lat0', 'lon0', 'xy_sd', 't_0'] if key in globals()}
packages = {}
for package in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'xgboost', 'lightgbm', 'pyswarm', 'cloudpickle']:
    try:
        packages[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        packages[package] = None

model_parameters = {name: model.get_params(deep=True) for name, model in models_tree.items()}
ridge_model = globals().get('rigde_full', globals().get('ridge_full'))
if ridge_model is not None:
    model_parameters['Ridge'] = ridge_model.get_params(deep=True)

gp_parameters = {}
missing_fitted = []
for name in models_tree:
    state = store.get(name, {})
    fitted_gp = state.get('gp')
    if state.get('tree') is None or fitted_gp is None:
        missing_fitted.append(name)
    if fitted_gp is not None:
        gp_parameters[name] = {
            'kernel': str(fitted_gp.kernel_),
            'theta_log': fitted_gp.kernel_.theta,
            'bounds_log': fitted_gp.kernel_.bounds,
            'fit_parameters': fitted_gp.get_params(deep=False),
            'log_marginal_likelihood': fitted_gp.log_marginal_likelihood_value_,
        }

# Ghi đúng kết quả tuning mô hình cuối, tách khỏi tuning của từng outer fold.
pso_parameters = {entry['model']: entry for entry in tuning_log
                  if entry['context'] == 'final_train'}

source_file = Path(str(globals().get('csv_file', '')))
source_hash = hashlib.sha256(source_file.read_bytes()).hexdigest() if source_file.is_file() else None
record = {
    'schema_version': 2, 'bundle_id': run_dir.name,
    'source_notebook': 'salinity-predict.ipynb',
    'notebook_template_code_sha256': NOTEBOOK_TEMPLATE_CODE_SHA256,
    'template_hash_note': 'Hash mẫu code đã phát hành; cấu hình thực chạy được lưu riêng trong config.',
    'saved_at': now.isoformat(),
    'config': {key: globals().get(key) for key in config_keys},
    'model_parameters': model_parameters,
    'pso_parameters_if_available': pso_parameters,
    'gp_parameters': gp_parameters,
    'blend_weights': {name: state.get('blend_weights') for name, state in store.items()},
    'base_cols': base_cols,
    'preprocessing': preprocessing,
    'cv_indices': oof_splits,
    'oof_method': 'nested_forward',
    'gp_residual_kind': BASE_MODE,
    'gp_kernel_structure': 'trend + season + space + noise',
    'split_dates': {'train_start': '2019-01-01', 'train_end_exclusive': '2020-03-01'},
    'nested_tuning_log': tuning_log, 'oof_fold_log': oof_fold_log,
    'data_shapes': {key: np.shape(globals()[key]) for key in ['X_train_base', 'X_test_base', 'y_train', 'y_test'] if key in globals()},
    'data_sha256': source_hash,
    'python_version': sys.version,
    'package_versions': packages,
    'missing_fitted_models': missing_fitted,
    'note': 'Nested forward OOF: outer validation không tham gia tuning/fit của fold; GP residual theo BASE_MODE.',
}
(run_dir / 'parameters.json').write_text(
    json.dumps(json_value(record), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8'
)
results_df.to_csv(run_dir / 'metrics.csv', encoding='utf-8-sig')

# Bảng đủ chiều dài train để nối chính xác với ngày/tọa độ và chia nhóm độ mặn dự báo.
oof_tables = []
for name in models_tree:
    state = store[name]
    frame = df_train[['source_row', 'date', 'Lat', 'Long']].copy()
    frame.insert(0, 'train_index', np.arange(len(y_train)))
    frame['model'] = name
    frame['y_true'] = y_train
    for key in ['oof_tree', 'oof_ridge', 'oof_blend', 'residual_oof',
                'residual_tree_oof', 'oof_w_tree', 'oof_fold_id']:
        frame[key] = state[key]
    frame['used_for_gp'] = state['oof_mask']
    oof_tables.append(frame)
pd.concat(oof_tables, ignore_index=True).to_csv(
    run_dir / 'oof_predictions.csv', index=False, encoding='utf-8-sig')
for filename, entries in [('nested_tuning_log.json', tuning_log), ('oof_fold_log.json', oof_fold_log)]:
    (run_dir / filename).write_text(
        json.dumps(json_value(entries), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

prediction_rows = []
for name, outputs in predictions.items():
    frame = pd.DataFrame({'y_true': np.asarray(y_test)})
    if 'df_test' in globals():
        for column in ['date', 'Lat', 'Long']:
            if column in df_test:
                frame[column] = df_test[column].to_numpy()
    frame['model'] = name
    for stage, values in outputs.items():
        frame[f'pred_{stage}'] = np.asarray(values)
    prediction_rows.append(frame)
if prediction_rows:
    pd.concat(prediction_rows, ignore_index=True).to_csv(
        run_dir / 'test_predictions.csv', index=False, encoding='utf-8-sig'
    )

# Giữ mô hình đã fit và lớp kernel tùy chỉnh để có thể nạp lại, không phải chạy lại PSO.
with (run_dir / 'fitted_models.pkl').open('wb') as file:
    cloudpickle.dump({
        'schema_version': 2, 'bundle_id': run_dir.name,
        'source_notebook': 'salinity-predict.ipynb', 'notebook_template_code_sha256': NOTEBOOK_TEMPLATE_CODE_SHA256,
        'gp_residual_kind': BASE_MODE, 'gp_kernel_structure': 'trend + season + space + noise',
        'store': store, 'ridge': ridge_model,
        'base_cols': base_cols, 'preprocessing': preprocessing,
        'oof_method': 'nested_forward', 'tuning_log': tuning_log, 'oof_fold_log': oof_fold_log,
    }, file)

# JSON ghi hash PKL để nhận biết bộ file có bị trộn giữa các lần chạy.
record['fitted_models_sha256'] = hashlib.sha256((run_dir / 'fitted_models.pkl').read_bytes()).hexdigest()
(run_dir / 'parameters.json').write_text(
    json.dumps(json_value(record), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

archive = shutil.make_archive(str(run_dir), 'zip', root_dir=run_dir)
print(f'Đã lưu: {run_dir}')
print(f'File ZIP để tải về: {archive}')
for filename in ['fitted_models.pkl', 'parameters.json', 'oof_predictions.csv']:
    print('  ', run_dir / filename)
print('Ba notebook thí nghiệm: dùng PKL và JSON trong cùng thư mục vừa lưu.')
if missing_fitted:
    print('Đã lưu tham số, nhưng thiếu cây/GP đã fit của:', missing_fitted)
    print("Cho lần train tiếp theo, thêm 'tree': tree_full và 'gp': gp vào store[name].")

## CNN tùy chọn
Mặc định tắt để tập trung bộ model cây + GP cần xuất. Nếu bật phải cài PyTorch trong môi trường riêng; CNN không thuộc bộ cây vừa lưu.

In [ ]:
%%gp312
# ----------------------------------------------------------------------
# 7.1 (TÙY CHỌN) 1D-CNN. Mặc định TẮT: tích chập trượt trên trục "đặc trưng" của dữ liệu bảng
# không có ý nghĩa vì thứ tự cột là tùy ý. Nếu bật, đầu vào được chuẩn hóa, early stopping dùng
# 15% cảnh muộn nhất trong tập huấn luyện, và GP học phần dư OOF (không phải in-sample).
# ----------------------------------------------------------------------
RUN_CNN = False

if RUN_CNN:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader

    class CNN1D_Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
            self.bn1 = nn.BatchNorm1d(16)
            self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
            self.bn2 = nn.BatchNorm1d(32)
            self.pool = nn.AdaptiveAvgPool1d(8)
            self.dropout = nn.Dropout(0.20)
            self.fc1 = nn.Linear(32 * 8, 64)
            self.fc2 = nn.Linear(64, 1)

        def forward(self, x):
            x = torch.relu(self.bn1(self.conv1(x)))
            x = torch.relu(self.bn2(self.conv2(x)))
            x = self.dropout(self.pool(x).flatten(1))
            return self.fc2(torch.relu(self.fc1(x)))

    def train_cnn(X_tr, y_tr, X_va, y_va, epochs=200, lr=0.001, batch_size=32, patience=25, seed=42):
        torch.manual_seed(seed)
        X_tr_t, y_tr_t = torch.FloatTensor(X_tr).unsqueeze(1), torch.FloatTensor(y_tr).unsqueeze(1)
        X_va_t, y_va_t = torch.FloatTensor(X_va).unsqueeze(1), torch.FloatTensor(y_va).unsqueeze(1)
        loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)
        model = CNN1D_Model()
        crit = nn.MSELoss()
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
        best, best_w, cnt = float('inf'), None, 0
        for ep in range(epochs):
            model.train()
            for bx, by in loader:
                opt.zero_grad()
                loss = crit(model(bx), by)
                if torch.isfinite(loss):
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
            model.eval()
            with torch.no_grad():
                vl = crit(model(X_va_t), y_va_t).item()
            if np.isfinite(vl):
                sched.step(vl)
                if vl < best:
                    best, best_w, cnt = vl, copy.deepcopy(model.state_dict()), 0
                else:
                    cnt += 1
            if cnt >= patience:
                break
        if best_w is not None:
            model.load_state_dict(best_w)
        return model

    train_dates = df_train['date'].values

    def cnn_fit_predict(tr_idx, X_new_list):
        sc = StandardScaler().fit(X_train_base[tr_idx])          # chuẩn hóa chỉ bằng dữ liệu huấn luyện
        d = train_dates[tr_idx]
        ud = np.unique(d)
        is_val = d >= ud[int(len(ud) * 0.85)]                      # 15% cảnh muộn nhất làm validation
        Xa, ya = sc.transform(X_train_base[tr_idx]), y_train[tr_idx]
        model = train_cnn(Xa[~is_val], ya[~is_val], Xa[is_val], ya[is_val])
        model.eval()
        with torch.no_grad():
            return [np.clip(model(torch.FloatTensor(sc.transform(X)).unsqueeze(1)).numpy().ravel(), 0, None)
                    for X in X_new_list]

    print('-> Đang huấn luyện 1D-CNN (OOF + full)...')
    cnn_oof = np.full(len(y_train), np.nan)
    for tr, va in oof_splits:
        cnn_oof[va] = cnn_fit_predict(tr, [X_train_base[va]])[0]
    cnn_pred_train, cnn_pred_test = cnn_fit_predict(np.arange(len(y_train)), [X_train_base, X_test_base])

    cnn_resid = y_train - cnn_oof
    cnn_valid = np.isfinite(cnn_oof)
    cnn_gps = fit_correction_joint(t_tr[cnn_valid], sc_tr[cnn_valid], xy_tr[cnn_valid], cnn_resid[cnn_valid])
    cnn_hybrid_te = np.clip(cnn_pred_test + predict_correction_joint(cnn_gps, t_te, sc_te, xy_te), 0, None)

    ev_tr, ev_cv = evaluate_model(y_train, cnn_pred_train), evaluate_oof(y_train, cnn_oof)
    results.append(row('1D-CNN', ev_tr, ev_cv, evaluate_model(y_test, cnn_pred_test)))
    results.append(row('1D-CNN + GP', ev_tr, ev_cv, evaluate_model(y_test, cnn_hybrid_te)))
    predictions['1D-CNN'] = {'base': cnn_pred_test, 'hybrid': cnn_hybrid_te}
    store['1D-CNN'] = {'pred_test_base': cnn_pred_test, 'residual_oof': cnn_resid}
else:
    print('RUN_CNN = False -> bỏ qua 1D-CNN.')

## Tổng hợp kết quả
Hiển thị lại bảng kết quả và xuất CSV để xem nhanh. Bộ model đã được lưu ở bước trước.

In [ ]:
%%gp312
# ----------------------------------------------------------------------
# 3. TỔNG HỢP VÀ HIỂN THỊ BẢNG KẾT QUẢ ĐỐI CHUẨN TOÀN DIỆN (đánh giá TĨNH)
# ----------------------------------------------------------------------
results_df = pd.DataFrame(results).set_index('Model')
print('=' * 95)
print('BẢNG SO SÁNH HIỆU NĂNG: đánh giá tĩnh (GP chỉ thấy train). So CV_R2 (OOF) với Test_R2.')
print('=' * 95)
display(results_df)

results_df.to_csv(f'result_hybrid_gp_{RUN_TAG}.csv')
print(f'-> Đã lưu: result_hybrid_gp_{RUN_TAG}.csv và result_rolling_gp_{RUN_TAG}.csv')

**Cell 7.3: Phân tích chỉ số Bias theo từng ngày & Đối chuẩn chi tiết PSO-RF vs Phương pháp mới**
- **Mục tiêu:** Theo đúng chỉ đạo của Thầy, đánh giá chi tiết độ lệch (Bias) qua từng cảnh ngày kiểm thử (từ 10/03/2020 đến 14/06/2020).
- **Công thức Bias:** $\text{Bias} = \frac{1}{N} \sum (y_{obs} - \hat{y}_{pred})$ (dương nghĩa là đoán thấp, âm nghĩa là đoán cao).
- **Tiêu chí thắng:** Phương pháp mới (+ GP) phải triệt tiêu độ lệch lớn ở các cảnh cực đoan, hạ thấp RMSE và MAE của từng ngày so với PSO-RF gốc.

In [ ]:
%%gp312
# ==============================================================================
# BẢNG ĐỐI CHUẨN VÀ BIỂU ĐỒ DAILY BIAS: PSO-RF (GỐC) VS PHƯƠNG PHÁP MỚI (+ GP)
# ==============================================================================
import matplotlib.pyplot as plt

# Lấy dự báo của PSO-RF gốc và Mô hình đề xuất
pso_rf_pred = predictions['PSO RF']['tree_only']
hybrid_pred = predictions['PSO LightGBM']['hybrid']

df_daily = pd.DataFrame({
    'date': df_test['date'],
    'y_true': y_test,
    'pred_rf': pso_rf_pred,
    'pred_hybrid': hybrid_pred
})
df_daily['date_str'] = df_daily['date'].dt.strftime('%d/%m/%Y')
df_daily['bias_rf'] = df_daily['y_true'] - df_daily['pred_rf']
df_daily['bias_hybrid'] = df_daily['y_true'] - df_daily['pred_hybrid']
df_daily['sq_err_rf'] = (df_daily['y_true'] - df_daily['pred_rf']) ** 2
df_daily['sq_err_hybrid'] = (df_daily['y_true'] - df_daily['pred_hybrid']) ** 2

daily_summary = df_daily.groupby('date_str', sort=False).agg(
    n_samples=('y_true', 'count'),
    mean_obs=('y_true', 'mean'),
    bias_rf=('bias_rf', 'mean'),
    bias_hybrid=('bias_hybrid', 'mean'),
    rmse_rf=('sq_err_rf', lambda s: np.sqrt(np.mean(s))),
    rmse_hybrid=('sq_err_hybrid', lambda s: np.sqrt(np.mean(s)))
).reset_index()

daily_summary['abs_bias_impr'] = daily_summary['bias_rf'].abs() - daily_summary['bias_hybrid'].abs()
daily_summary['rmse_impr'] = daily_summary['rmse_rf'] - daily_summary['rmse_hybrid']

print('=' * 105)
print('BẢNG ĐỐI CHUẨN CHI TIẾT THEO TỪNG CẢNH NGÀY (DAILY BIAS & RMSE) TRÊN TẬP TEST 2020')
print('=' * 105)
display_df = daily_summary.rename(columns={
    'date_str': 'Ngày chụp',
    'n_samples': 'Số trạm',
    'mean_obs': 'Độ mặn TB (‰)',
    'bias_rf': 'Bias PSO-RF (‰)',
    'bias_hybrid': 'Bias Mới + GP (‰)',
    'rmse_rf': 'RMSE PSO-RF (‰)',
    'rmse_hybrid': 'RMSE Mới + GP (‰)',
    'abs_bias_impr': 'Cải thiện |Bias| (‰)',
    'rmse_impr': 'Cải thiện RMSE (‰)'
})
display(display_df.round(3))

# Vẽ biểu đồ cột đối chuẩn trực quan Daily Bias & Daily RMSE
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x_indices = np.arange(len(daily_summary))
width = 0.35

# 1. Subplot Bias
axes[0].bar(x_indices - width/2, daily_summary['bias_rf'], width, label='PSO-RF', color='#e74c3c', alpha=0.85, edgecolor='black')
axes[0].bar(x_indices + width/2, daily_summary['bias_hybrid'], width, label='PSO LightGBM + GP', color='#2ecc71', alpha=0.85, edgecolor='black')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_xticks(x_indices)
axes[0].set_xticklabels(daily_summary['date_str'], rotation=30, ha='right', fontsize=11)
axes[0].set_ylabel('Bias Độ mặn (‰)', fontsize=12)
axes[0].set_title('So sánh Bias theo từng ngày chụp', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', linestyle=':', alpha=0.7)

# 2. Subplot RMSE
axes[1].bar(x_indices - width/2, daily_summary['rmse_rf'], width, label='PSO-RF', color='#e74c3c', alpha=0.85, edgecolor='black')
axes[1].bar(x_indices + width/2, daily_summary['rmse_hybrid'], width, label='PSO LightGBM + GP', color="#2ecc71", alpha=0.85, edgecolor='black')
axes[1].set_xticks(x_indices)
axes[1].set_xticklabels(daily_summary['date_str'], rotation=30, ha='right', fontsize=11)
axes[1].set_ylabel('RMSE Độ mặn (‰)', fontsize=12)
axes[1].set_title('So sánh RMSE theo từng ngày chụp', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.savefig('daily_bias_pso_rf_vs_hybrid.png', dpi=300, bbox_inches='tight')
plt.show()
print('-> Đã lưu biểu đồ so sánh Daily Bias: daily_bias_pso_rf_vs_hybrid.png')
daily_summary.to_csv(f'result_daily_bias_{RUN_TAG}.csv', index=False)


**Cell 8: Trực quan hóa kết quả dự báo (Scatter Plot 1:1 & Phân tích sai số Residual)**
- Biểu đồ Scatter Plot phân tán (Observed vs Predicted) so sánh trực tiếp mô hình Đơn lẻ vs Mô hình Lai (+ GP).
- Đánh giá trực quan mức độ cải thiện và khả năng hội tụ dọc theo đường chéo 1:1.

In [ ]:
%%gp312
# Chọn mô hình để trực quan hóa
target_model = 'PSO LightGBM'
pred_te_pure   = predictions[target_model].get('tree_only', predictions[target_model]['base'])
pred_te_base   = predictions[target_model]['base']
pred_te_hybrid = predictions[target_model]['hybrid']
pred_te_roll   = predictions[target_model].get('rolling', pred_te_hybrid)

eval_pure = evaluate_model(y_test, pred_te_pure)
eval_base = evaluate_model(y_test, pred_te_base)
eval_hybr = evaluate_model(y_test, pred_te_hybrid)

panels = [
    (pred_te_pure,   f'{target_model}', eval_pure, '#e74c3c'),
    (pred_te_hybrid, f'{target_model} + GP', eval_hybr, '#2ca02c'),
]
fig, axes = plt.subplots(1, 2, figsize=(21, 6))
max_val = max(y_test.max(), *(p.max() for p, *_ in panels)) + 2
for ax, (p, title, ev, col) in zip(axes, panels):
    ax.scatter(y_test, p, alpha=0.6, edgecolors='k', linewidths=0.5, c=col)
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='1:1 Line')
    ax.set_xlabel('Observed Salinity (‰)', fontsize=12)
    ax.set_ylabel('Predicted Salinity (‰)', fontsize=12)
    ax.set_title(f'{title}\nR² = {ev["R2"]}, RMSE = {ev["RMSE (‰)"]} ‰, MAE = {ev["MAE (‰)"]} ‰, Bias = {ev["Bias (‰)"]}', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('scatter_plot_hybrid_gp_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('-> Đã lưu biểu đồ so sánh: scatter_plot_hybrid_gp_comparison.png')


**Cell 9: Ablation – mô hình đang dựa vào những nhóm biến nào?**
Bỏ lần lượt `sin_doy/cos_doy` và `Lat/Long` rồi chạy lại một mô hình (PSO LightGBM, không blend, không GP) để xem R² và bias thay đổi ra sao. Nếu bỏ `sin/cos_doy` mà bias giảm, mô hình đang bám vào mùa của năm 2019.

Đây là thăm dò trên test đã xem; không dùng kết quả này để chọn cấu hình rồi vẫn gọi cùng tập đó là test độc lập.

In [ ]:
%%gp312
print('Cờ USE_SEASONAL hiện tại =', USE_SEASONAL)

def eval_drop(drop, model=pso_lgbm_model):
    keep = [i for i, f in enumerate(base_cols) if f not in drop]
    m = clone(model).fit(X_train_base[:, keep], y_train)
    ev = evaluate_model(y_test, m.predict(X_test_base[:, keep]))
    return ev['R2'], ev['RMSE (‰)'], ev['Bias (‰)']

ablation_cases = {
    'Đầy đủ (38 biến)': [],
    'Bỏ sin/cos doy (mùa vụ)': ['sin_doy', 'cos_doy'],
    'Bỏ số ngày tuyệt đối t (dài hạn)': ['t'],
    'Bỏ cả t và sin/cos doy': ['t', 'sin_doy', 'cos_doy'],
    'Bỏ tọa độ Lat/Long': ['Lat', 'Long'],
    'Chỉ giữ 35 biến viễn thám': ['t', 'sin_doy', 'cos_doy', 'Lat', 'Long']
}
print(f'{"Cấu hình đặc trưng":38s} {"R2":>7s} {"RMSE (‰)":>10s} {"Bias (‰)":>10s}')
print('-' * 70)
for label, drop in ablation_cases.items():
    r2, rmse, bias = eval_drop(drop)
    print(f'{label:38s} {r2:7.3f} {rmse:10.3f} {bias:10.3f}')


## Dùng bộ model mới trong các notebook thí nghiệm

1. Sau Run All, tải ZIP trong `/kaggle/working/training_runs/`. PKL, `parameters.json`, `oof_predictions.csv` và hai log fold nằm cùng thư mục.
2. Tạo một Kaggle Dataset chứa PKL và JSON của **cùng lần train**; thêm OOF CSV nếu muốn xem bảng dự báo. Add Input dataset này và CSV gốc vào từng notebook thí nghiệm.
3. Ở mục 0.1 của notebook thí nghiệm, đặt `GP312_PARAMETERS_JSON` trỏ đến JSON mới. Ở mục đường dẫn, đặt `MODEL_PKL_PATH` trỏ đến PKL cùng bộ. Nếu có nhiều file trùng tên, luôn khai báo đường dẫn đầy đủ.
4. Chạy từ đầu trong một session mới, không dùng checkpoint/output của bộ model cũ. Không cần chạy lại PSO/cây trong ba notebook thí nghiệm; chúng đọc trực tiếp `store[name]['oof_tree']`, nguồn là `saved_oof_tree`.
5. Mặc định GP đã lưu khớp residual cây và chỉ có kernel cộng cơ sở. Lần kernel thêm tương tác; lần tổng hợp học lại GP rồi chọn lambda trên validation tháng 03–04/2020, chấm test từ tháng 05/2020.
6. `parameters.json` ghi phiên bản môi trường, hash CSV/PKL, cấu hình, chỉ số inner/outer fold và mã định danh lần chạy để đối chiếu. Mỗi lần chạy phải tải cả bộ, tránh trộn PKL mới với JSON cũ.

Nested OOF đánh giá quy trình chọn tham số trên các giai đoạn quá khứ; mô hình cuối có thể có tham số khác từng outer fold. Test gốc đã được xem trong các thí nghiệm trước, nên kết quả chạy lại là đánh giá hồi cứu.
